In [1]:
# importing libraries

import re 
import pandas as pd
import numpy as np

In [2]:
# load the file here
# Set the path of the foalder, because of there are four sheets.
data_path = "D:/D/Projects/Healthcare Analytics/data/raw"

# Load excel files
admissions_file = pd.read_csv(data_path+"/admissions.csv")
claims_file = pd.read_csv(data_path+"/claims.csv")
patients_file = pd.read_csv(data_path+"/patients.csv")
readmissions_file = pd.read_csv(data_path+"/readmissions.csv")

In [3]:
# finding shape of each tables
# Why this matters professionally:

# Row counts confirm:
# dataset generated correctly
# no file truncation happened
# joins later won’t silently fail
table_name = ['admission_file', 'claims_file', 'patients_file', 'readmissions_file']
tables = [admissions_file, claims_file, patients_file, readmissions_file]

def tables_shape():
    for name, file in zip(table_name, tables):
        shape = file.shape
        print(name + ': ', shape)

tables_shape()

admission_file:  (200000, 11)
claims_file:  (200000, 9)
patients_file:  (10200, 10)
readmissions_file:  (3000, 8)


In [4]:
# Clumn overview
# Finding information of sheets which tells missing value, data types, column presence, and memory footprint
def tables_info():
    for name, file in zip(table_name, tables):
        print("Information for the " + name)
        info = file.info()

tables_info()

Information for the admission_file
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   admission_id         200000 non-null  object
 1   patient_id           200000 non-null  object
 2   admit_date           200000 non-null  object
 3   discharge_date       200000 non-null  object
 4   department           194000 non-null  object
 5   diagnosis_code       199488 non-null  object
 6   diagnosis_desc       200000 non-null  object
 7   severity             200000 non-null  object
 8   attending_physician  200000 non-null  object
 9   los_days             200000 non-null  int64 
 10  outcome              200000 non-null  object
dtypes: int64(1), object(10)
memory usage: 16.8+ MB
Information for the claims_file
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200000 entries, 0 to 199999
Data columns (total 9 columns):
 #   Colum

In [5]:
# CHECKING PRIMARY KEYS
# In relational dataset, primary key is essential part of the data table. It must always be unique. 

primary_key = [admissions_file['admission_id'].is_unique, 
               claims_file['claim_id'].is_unique,
               patients_file['patient_id'].is_unique,
               readmissions_file['readmit_id'].is_unique]

for result in primary_key:
    print(result)

True
True
True
True


In [6]:
# DUPLICATION DETECTION
# 1. Admissions file
patients_file.duplicated(subset=['full_name', 'dob']).sum()

np.int64(200)

In [7]:
# NULL VALUE SCAN
# This identifies:which columns are damaged and how badly they are damaged
def tables_null_value():
    for name, file in zip(table_name, tables):
        print("Information for ", name)
        result = file.isnull().sum()
        print(result)
        print(" *************************************** ")
tables_null_value()

Information for  admission_file
admission_id              0
patient_id                0
admit_date                0
discharge_date            0
department             6000
diagnosis_code          512
diagnosis_desc            0
severity                  0
attending_physician       0
los_days                  0
outcome                   0
dtype: int64
 *************************************** 
Information for  claims_file
claim_id                0
admission_id            0
billed_amount           0
approved_amount         0
payer                   0
claim_status         4000
denial_reason      159963
submission_date         0
payment_date        99789
dtype: int64
 *************************************** 
Information for  patients_file
patient_id        0
full_name         0
dob               0
gender            0
blood_type        0
insurance_type    0
region            0
signup_date       0
phone             0
email             0
dtype: int64
 *************************************** 
I

In [8]:
# Finding perc of null values in table
def tables_null_percentage():
    for name, file in zip(table_name, tables):
        print("PERCENTAGE -",name)
        result = (file.isnull().mean()*100).round(2)
        print(result)
        print("")
tables_null_percentage()

PERCENTAGE - admission_file
admission_id           0.00
patient_id             0.00
admit_date             0.00
discharge_date         0.00
department             3.00
diagnosis_code         0.26
diagnosis_desc         0.00
severity               0.00
attending_physician    0.00
los_days               0.00
outcome                0.00
dtype: float64

PERCENTAGE - claims_file
claim_id            0.00
admission_id        0.00
billed_amount       0.00
approved_amount     0.00
payer               0.00
claim_status        2.00
denial_reason      79.98
submission_date     0.00
payment_date       49.89
dtype: float64

PERCENTAGE - patients_file
patient_id        0.0
full_name         0.0
dob               0.0
gender            0.0
blood_type        0.0
insurance_type    0.0
region            0.0
signup_date       0.0
phone             0.0
email             0.0
dtype: float64

PERCENTAGE - readmissions_file
readmit_id               0.0
patient_id               0.0
original_admission_id    0.0
r

In [9]:
# DATE LOGIC VALIDATION
# This counts invalid hospital stays. These are high-risk errors in healthcare analytics.
def tables_date_logic_validation():
    admissions_file['admit_date'] = pd.to_datetime(admissions_file['admit_date'])
    admissions_file['discharge_date'] = pd.to_datetime(admissions_file['discharge_date'])
    result = (admissions_file['discharge_date'] < admissions_file['admit_date']).sum()
    print(result)
tables_date_logic_validation()


2000


In [10]:
# NEGATIVE VALUE DETACTION
# Negative LOS = impossible medically
# Negative billing = impossible financially
# These must be flagged.
print((admissions_file['los_days'] < 0).sum())
print((claims_file['billed_amount'] < 0).sum())


2000
2000


In [11]:
# REFERENTIAL INTEGRITY
# referentail integrity is a relationa dataset concept, which insures that relationship between table_name remain constent and accurate.
refrential_intg = [(~admissions_file['patient_id'].isin(patients_file['patient_id'])).sum(),
                   (~claims_file['admission_id'].isin(admissions_file['admission_id'])).sum(),
                   (~readmissions_file['patient_id'].isin(patients_file['patient_id'])).sum(),
                   (~readmissions_file['original_admission_id'].isin(admissions_file['admission_id'])).sum()
                   ]
refrential_intg

[np.int64(1000), np.int64(0), np.int64(19), np.int64(0)]

INSPECT DIAGNOSIS CLUMNS


In [12]:
# Inspect Diagnosis column first
# Main reason behind this we will confirm that column is exist, see raw patterns, and catch weird spacing and formating issues
admissions_file[['diagnosis_code', 'diagnosis_desc']].head(10)

,diagnosis_code,diagnosis_desc
0,E11.9,Type 2 diabetes without complications
1,N18.3,"Chronic kidney disease, stage 3"
2,S72.001,Femur fracture
3,J44.1,COPD with acute exacerbation
4,G35,Multiple sclerosis
5,S72.001,Femur fracture
6,I63.9,"Cerebral infarction, unspecified"
7,K92.1,Melena
8,E11.9,Type 2 diabetes without complications
9,I63.9,"Cerebral infarction, unspecified"


In [13]:
# count missing diagnosis values in percentages
round(admissions_file[['diagnosis_code', 'diagnosis_desc']].isna().mean() * 100, 2)

diagnosis_code    0.26
diagnosis_desc    0.00
dtype: float64

In [14]:
# Perform standardization in diagnosis column, because diagnosis_code contains space, unwanted chaacter, and formatting issues
admissions_file["diagnosis_code_clean"] = (
        admissions_file['diagnosis_code']
        .astype("string")
        .str.strip()
        .str.upper()
)


In [15]:
# Appply ICD style formate validation
icd_pattern = r"^[A-Z][0-9]{2}(\.[A-Z0-9]{1,3})?$"
admissions_file['icd_format_valid'] = admissions_file['diagnosis_code_clean'].str.match(
    icd_pattern,
    na=False
)

In [16]:
invalid_icd_count = (~admissions_file["icd_format_valid"]).sum()
invalid_icd_pct = round((~admissions_file["icd_format_valid"]).mean() * 100, 2)

print("Invalid ICD format count:", invalid_icd_count)
print("Invalid ICD format %:", invalid_icd_pct)

Invalid ICD format count: 3512
Invalid ICD format %: 1.76


In [17]:
# Inspect valid example
admissions_file.loc[
    ~admissions_file['icd_format_valid'],['diagnosis_code', 'diagnosis_code_clean', 'diagnosis_desc']
].drop_duplicates().head(5)

,diagnosis_code,diagnosis_code_clean,diagnosis_desc
174,123AB,123AB,INVALID
207,MISSING,MISSING,INVALID
246,ZZZ00,ZZZ00,INVALID
374,999.99,999.99,INVALID
667,NaN,<NA>,INVALID


In [18]:
# Sperater missing and malformed data
admissions_file['diagnosis_code_missing_flag'] = admissions_file['diagnosis_code_clean'].isna()
admissions_file['diagnosis_code_invalid_format_flag'] = (
    ~admissions_file['diagnosis_code_missing_flag'] & ~admissions_file['icd_format_valid']
)

print("Missing diagnosis code: ", admissions_file['diagnosis_code_missing_flag'].sum())
print("Malformed diagnosis codes: ", admissions_file['diagnosis_code_invalid_format_flag'].sum())

Missing diagnosis code:  512
Malformed diagnosis codes:  3000


In [19]:
# we performed those for code only, Now preparing for description to.
admissions_file['diagnosis_desc_clean'] = (
    admissions_file['diagnosis_desc']
    .astype("string")
    .str.strip()
    .str.upper()
)

# flag suspicious description
admissions_file['diagnosis_desc_invalid_flag'] = admissions_file['diagnosis_desc_clean'].isin(["INVALID", "N/A", "MISSING", ""])

admissions_file.loc[
    admissions_file["diagnosis_desc_invalid_flag"],
    ['diagnosis_code', 'diagnosis_desc']
].drop_duplicates().head(20)

,diagnosis_code,diagnosis_desc
174,123AB,INVALID
207,MISSING,INVALID
246,ZZZ00,INVALID
374,999.99,INVALID
386,I21,INVALID
667,NaN,INVALID
1142,XX999,INVALID
1530,BAD-CODE,INVALID


In [20]:
# build a final diagnosis quality status column
admissions_file['diagnosis_quality_status'] = np.select(
    [
        admissions_file['diagnosis_code_missing_flag'],
        admissions_file['diagnosis_code_invalid_format_flag'],
        admissions_file['diagnosis_desc_invalid_flag']
    ],
    [
        "Missing diagnosis code",
        "Malformed diagnosis code",
        "Suspicious diagnosis description"
    ],
    default="Valid"
)
admissions_file['diagnosis_quality_status'].value_counts()

diagnosis_quality_status
Valid                               196000
Malformed diagnosis code              3000
Missing diagnosis code                 512
Suspicious diagnosis description       488
Name: count, dtype: int64

In [21]:
admissions_file.head(2
                     )

,admission_id,patient_id,admit_date,discharge_date,department,diagnosis_code,diagnosis_desc,severity,attending_physician,los_days,outcome,diagnosis_code_clean,icd_format_valid,diagnosis_code_missing_flag,diagnosis_code_invalid_format_flag,diagnosis_desc_clean,diagnosis_desc_invalid_flag,diagnosis_quality_status
0,ADM-000001,PAT-01883,2021-06-02,2021-06-20,NaN,E11.9,Type 2 diabetes without complications,Low,Robert Ibarra,18,AMA,E11.9,True,False,False,TYPE 2 DIABETES WITHOUT COMPLICATIONS,False,Valid
1,ADM-000002,PAT-02083,2023-09-12,2023-09-18,Oncology,N18.3,"Chronic kidney disease, stage 3",Critical,Katherine Byrd,6,Discharged,N18.3,True,False,False,"CHRONIC KIDNEY DISEASE, STAGE 3",False,Valid


In [22]:
# Quantify how much of admissions data is unrealiable for diagnosis.
diagnosis_unstable_count = (
    admissions_file['diagnosis_quality_status'] != "Valid"
).sum()

diagnosis_unstable_pct = round(
    (admissions_file['diagnosis_quality_status'] != 'Valid').mean() * 100,
    2
)

print("Diagnosis-unstable admission rows: ", diagnosis_unstable_count)
print("Diagnosis-unstable %: ", diagnosis_unstable_pct)

Diagnosis-unstable admission rows:  4000
Diagnosis-unstable %:  2.0


In [23]:
# Create a profiling summary row for ICD validation
icd_summary = pd.DataFrame(
    {
        "check_name": ["invalid_diagnosis_code_format", "missing_diagnosis_code", "suspicious_diagnosis_desc"],
        "table_name": ["admissions", "admissions", "admissions"],
        "issue_count": [
            admissions_file['diagnosis_code_invalid_format_flag'].sum(),
            admissions_file['diagnosis_code_missing_flag'].sum(),
            admissions_file['diagnosis_desc_invalid_flag'].sum()
            ],
        "issue_pct": [
            round(admissions_file['diagnosis_code_invalid_format_flag'].mean()*100, 2),
            round(admissions_file['diagnosis_code_missing_flag'].mean()*100, 2),
            round(admissions_file['diagnosis_desc_invalid_flag'].mean()*100, 2)
            ],
        "severity": ['high', 'high', 'medium'],
        "recommended_action": [
            "Flag and exclude from diagnosis-based analysis; evaluate repair rules",
            "Mark as missing and exclude from diagnosis-based analysis",
            "Review with diagnosis code; recode as invalid if needed"
            ]
    }
)

icd_summary

,check_name,table_name,issue_count,issue_pct,severity,recommended_action
0,invalid_diagnosis_code_format,admissions,3000,1.50,high,Flag and exclude from diagnosis-based analysis...
1,missing_diagnosis_code,admissions,512,0.26,high,Mark as missing and exclude from diagnosis-bas...
2,suspicious_diagnosis_desc,admissions,4000,2.00,medium,Review with diagnosis code; recode as invalid ...


LOS CONSISTENCY VALIDATION

In [24]:
# LOS consistency validation
computed_los = (
    admissions_file['discharge_date'] - admissions_file['admit_date']
).dt.days

# compare compute_los and los_day to count total row which have different los
los_mismatch_count = (
    admissions_file['los_days'] != computed_los
).sum()

# find percentage for mismatch los days
los_mismatch_pct = round(
    (admissions_file['los_days'] != computed_los).mean() * 100,
    2
)

print("LOS mismatch count: ", los_mismatch_count)
print("LOS mismatch %: ", los_mismatch_pct)

LOS mismatch count:  3974
LOS mismatch %:  1.99


In [25]:
# Inspect the los's data
admissions_file.loc[
    admissions_file['los_days'] != computed_los,
    ["admit_date", "discharge_date", "los_days"]
]


,admit_date,discharge_date,los_days
7,2023-11-26,2023-11-21,30
37,2020-04-07,2020-05-02,-25
100,2022-03-13,2022-03-17,-4
101,2023-07-26,2023-08-10,-15
198,2021-12-05,2021-11-30,28
...,...,...,...
199435,2021-08-14,2021-08-10,15
199481,2023-07-03,2023-06-29,9
199549,2020-05-29,2020-06-03,-5
199672,2022-08-11,2022-08-02,17


In [26]:
# Identify reverse date logic cases
date_logic_error_count = (
    admissions_file['admit_date'] > admissions_file['discharge_date']
).sum()

# Identify reverse date logic in percentages
date_logic_error_pc = round((
    admissions_file['admit_date'] > admissions_file['discharge_date']
    ).mean() * 100,
    2
)

print("Negative los in csv file: ", (admissions_file['los_days']<0).sum())
print("Discharge before admit count: ", date_logic_error_count)
print("Discharge before admit %: ", date_logic_error_pc)

Negative los in csv file:  2000
Discharge before admit count:  2000
Discharge before admit %:  1.0


In [27]:
# Quantify the negative values
# Negative los flag for the already negative los days in file
admissions_file['negative_los_flag'] = admissions_file['los_days'] < 0

# date logic error flag was for checking mismatch date for discharge date and admit date
admissions_file['date_logic_error_flag'] = (
    admissions_file['discharge_date'] < admissions_file['admit_date']
)

print("Negative los flag: ", admissions_file['negative_los_flag'].sum())
print("Date Logic Error Flag: ", admissions_file['date_logic_error_flag'].sum())

Negative los flag:  2000
Date Logic Error Flag:  2000


In [28]:
#  QUESTIONS THOSE ASKED BY PROFESSIONAL IF FOUND ERROR IN LOS DATA
# How many?
# Which tables depend on this?
# Does LOS column match computed LOS?
# Is this isolated or systemic?
# Can we repair it safely?

In [29]:
# Orphan patients count
orphan_admission_count = (
    ~admissions_file['patient_id'].isin(patients_file['patient_id'])
).sum()

orphan_admission_per = round(
    (~admissions_file['patient_id'].isin(patients_file['patient_id'])).mean()*100,
    2
)

print("Orphan admission: ", orphan_admission_count)
print("Orphan admission %: ", orphan_admission_per)

Orphan admission:  1000
Orphan admission %:  0.5


FINANCIAL INTEGRITY CHECK

In [30]:
# Financial integrity check: aprroved amount should not exceed billed amount
finacial_error_count = (
    claims_file['approved_amount'] > claims_file['billed_amount']
).sum()

finacial_error_pct = round(
    (claims_file['approved_amount'] > claims_file['billed_amount']).mean() * 100,
    2
)

print("Approved > Billed amount: ", finacial_error_count)
print("Approved > Billed amount %: ", finacial_error_pct)

Approved > Billed amount:  4967
Approved > Billed amount %:  2.48


In [31]:
# Payment timeline validation
claims_file['payment_date'] = pd.to_datetime(claims_file['payment_date'])
claims_file['submission_date'] = pd.to_datetime(claims_file['submission_date'])

payment_date_error_count = (
    claims_file['payment_date'] < claims_file['submission_date']
).sum()

payment_date_error_per = round(
    (claims_file['payment_date'] < claims_file['submission_date']).mean() * 100,
    2
)

print("Payment before submission count: ", payment_date_error_count)
print("Payment before submission %: ", payment_date_error_per)

Payment before submission count:  501
Payment before submission %:  0.25


SUMMARY OF THE ANALYSIS

In [32]:
data_quality_summary = pd.DataFrame([
    ["Malformed ICD codes", "admissions", 3000, 1.50, "High"],
    ["Missing diagnosis codes", "admissions", 512, 0.26, "Medium"],
    ["Suspicious diagnosis descriptions", "admissions", 488, 0.24, "Medium"],
    ["Negative LOS", "admissions", 2000, 1.00, "Critical"],
    ["LOS mismatch", "admissions", 3974, 1.99, "High"],
    ["Orphan admissions", "admissions", 1000, 0.50, "Critical"],
    ["Approved > billed", "claims", 4967, 2.48, "Critical"],
    ["Payment before submission", "claims", 501, 0.25, "Medium"]
],
columns=["Issue", "Table", "Count", "% Impact", "Severity"])

data_quality_summary

,Issue,Table,Count,% Impact,Severity
0,Malformed ICD codes,admissions,3000,1.50,High
1,Missing diagnosis codes,admissions,512,0.26,Medium
2,Suspicious diagnosis descriptions,admissions,488,0.24,Medium
3,Negative LOS,admissions,2000,1.00,Critical
4,LOS mismatch,admissions,3974,1.99,High
5,Orphan admissions,admissions,1000,0.50,Critical
6,Approved > billed,claims,4967,2.48,Critical
7,Payment before submission,claims,501,0.25,Medium
